# 🇹🇷 Turkish LLM Model Merging — Full Pipeline

Bu notebook, Türkçe açık kaynak LLM modellerini **SLERP**, **TIES** ve **DARE** merge stratejileriyle birleştirir, benchmark ile karşılaştırır ve HuggingFace'e yükler.

**Modeller:**
- Model A: `ytu-ce-cosmos/Turkish-Llama-8b-Instruct-v0.1`
- Model B: `Trendyol/Trendyol-LLM-8b-chat-v2.0`
- Model C: `malhajar/Mistral-7b-tr` (sadece TIES/DARE)
- Base: `meta-llama/Meta-Llama-3-8B`

**⚠️ Gereksinimler:**
- Google Colab **T4 GPU** (ücretsiz tier yeterli)
- HuggingFace token (Colab Secrets'ta `HF_TOKEN` olarak kaydedin)
- GitHub Personal Access Token (Colab Secrets'ta `GITHUB_TOKEN` olarak kaydedin)

---

## [Section 0] Kurulum
⏱ Bu section ~5 dakika sürer

In [ ]:
# Gerekli kütüphaneleri kur
!pip install -q mergekit>=0.0.4 \
    transformers>=4.40.0 \
    accelerate>=0.28.0 \
    bitsandbytes>=0.43.0 \
    huggingface_hub>=0.22.0 \
    datasets>=2.18.0 \
    pyyaml>=6.0 \
    sentencepiece>=0.2.0 \
    protobuf>=4.25.0 \
    tqdm>=4.66.0 \
    tabulate>=0.9.0

print("✅ Tüm kütüphaneler kuruldu.")

In [ ]:
# Google Drive mount (opsiyonel ama önerilen — modelleri saklamak için)
from google.colab import drive

MOUNT_DRIVE = True  # False yaparsanız Drive mount edilmez

if MOUNT_DRIVE:
    drive.mount('/content/drive')
    # Merge çıktıları için klasör oluştur
    !mkdir -p /content/drive/MyDrive/turkish_merges/{slerp,ties,dare}
    print("✅ Google Drive bağlandı ve klasörler oluşturuldu.")
else:
    print("ℹ️  Drive mount atlandı. Modeller yalnızca Colab'da kalacak.")

In [ ]:
# HuggingFace Token + GitHub Token ayarla (Colab Secrets'tan)
import os

# ── HF_TOKEN ──
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    print("✅ HF_TOKEN Colab Secrets'tan okundu.")
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
    if HF_TOKEN:
        print("✅ HF_TOKEN ortam değişkeninden okundu.")
    else:
        print("⚠️  HF_TOKEN bulunamadı!")
        print("    Colab Secrets'a 'HF_TOKEN' olarak ekleyin.")
        print("    Veya: os.environ['HF_TOKEN'] = 'hf_...'")

# ── GITHUB_TOKEN ──
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN
    print("✅ GITHUB_TOKEN Colab Secrets'tan okundu.")
except Exception:
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '')
    if GITHUB_TOKEN:
        print("✅ GITHUB_TOKEN ortam değişkeninden okundu.")
    else:
        print("⚠️  GITHUB_TOKEN bulunamadı!")
        print("    Colab Secrets'a 'GITHUB_TOKEN' olarak ekleyin.")
        print("    GitHub → Settings → Developer settings → Personal access tokens → Fine-grained tokens")
        print("    Gerekli izin: Contents (read)")
        print("    Veya: os.environ['GITHUB_TOKEN'] = 'ghp_...'")

In [ ]:
# HuggingFace'e giriş yap
from huggingface_hub import login
login(token=HF_TOKEN)
print("✅ HuggingFace'e giriş yapıldı.")

In [ ]:
# Projeyi GitHub'dan klonla (GITHUB_TOKEN ile authenticated)
import os

GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '')
REPO_OWNER = 'Cosmobillian'
REPO_NAME = 'turkish-llm-merging'
CLONE_DIR = f'/content/{REPO_NAME}'
PROJECT_DIR = CLONE_DIR  # Diğer hücrelerde kullanmak için global

if GITHUB_TOKEN:
    clone_url = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
    print('🔑 GitHub token ile authenticated clone yapılıyor...')
else:
    clone_url = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'
    print('ℹ️  Token yok, public olarak klonlanıyor...')

if not os.path.exists(CLONE_DIR):
    !git clone {clone_url} {CLONE_DIR}
else:
    print(f'📂 {CLONE_DIR} zaten mevcut, pull yapılıyor...')
    !cd {CLONE_DIR} && git pull

%cd {CLONE_DIR}
print(f'\n📂 Çalışma dizini: {os.getcwd()}')

# Doğrulama: scripts klasörü var mı?
assert os.path.isdir('scripts'), f'❌ scripts/ klasörü bulunamadı! CWD={os.getcwd()}'
assert os.path.isdir('configs'), f'❌ configs/ klasörü bulunamadı! CWD={os.getcwd()}'
print('✅ Proje klonlandı ve doğrulandı.')
!ls -la

---
## [Section 1] Tokenizer Kontrolü
⏱ Bu section ~3 dakika sürer

In [ ]:
# Çalışma dizinini garantile
%cd {PROJECT_DIR}

# Tokenizer uyumluluk kontrolü
!python scripts/check_tokenizers.py

In [ ]:
# Sonuçları tablo olarak göster
import pandas as pd
from transformers import AutoTokenizer

models = [
    "ytu-ce-cosmos/Turkish-Llama-8b-Instruct-v0.1",
    "Trendyol/Trendyol-LLM-8b-chat-v2.0",
    "malhajar/Mistral-7b-tr",
]

info_list = []
for m in models:
    try:
        tok = AutoTokenizer.from_pretrained(m, trust_remote_code=True)
        info_list.append({
            "Model": m.split('/')[-1],
            "Vocab Size": tok.vocab_size,
            "Tokenizer": tok.__class__.__name__,
            "Padding Side": getattr(tok, 'padding_side', 'N/A'),
            "BOS Token": tok.bos_token,
            "EOS Token": tok.eos_token,
        })
    except Exception as e:
        print(f"⚠️  {m}: {e}")

df = pd.DataFrame(info_list)
display(df.style.set_caption("Tokenizer Karşılaştırması"))

# Uyumsuzluk uyarısı
if len(set(r['Vocab Size'] for r in info_list)) > 1:
    print("\n⚠️  UYARI: Vocab boyutları farklı!")
    print("   Bu durum SLERP merge'de sorun oluşturabilir.")
    print("   TIES/DARE base model kullandığı için daha toleranslıdır.")
    print("   Devam etmek istiyor musunuz?")
else:
    print("\n✅ Tüm tokenizer'lar uyumlu görünüyor.")

---
## [Section 2] SLERP Merge
⏱ Bu section ~45 dakika sürer (T4 GPU)

SLERP (Spherical Linear Interpolation) ile iki model birleştirilir:
- Model A: `ytu-ce-cosmos/Turkish-Llama-8b-Instruct-v0.1`
- Model B: `Trendyol/Trendyol-LLM-8b-chat-v2.0`
- `t=0.5` (eşit ağırlık)

In [ ]:
# Çalışma dizinini garantile
%cd {PROJECT_DIR}

# SLERP merge çalıştır
!python scripts/run_merge.py --strategy slerp

In [ ]:
# SLERP modelini Drive'a kopyala
if MOUNT_DRIVE:
    import shutil
    from tqdm import tqdm
    import os

    src = f"{PROJECT_DIR}/merged_models/slerp"
    dst = "/content/drive/MyDrive/turkish_merges/slerp"

    if os.path.exists(src):
        files = [f for f in os.listdir(src) if os.path.isfile(os.path.join(src, f))]
        for f in tqdm(files, desc="SLERP → Drive"):
            shutil.copy2(os.path.join(src, f), os.path.join(dst, f))
        print(f"✅ SLERP modeli Drive'a kopyalandı: {dst}")
    else:
        print(f"❌ Kaynak bulunamadı: {src}")
else:
    print("ℹ️  Drive mount edilmediği için kopyalama atlandı.")

---
## [Section 3] TIES Merge
⏱ Bu section ~60 dakika sürer (T4 GPU)

TIES (TrIm, Elect, Sum) ile üç model birleştirilir:
- Weights: [0.5, 0.3, 0.2]
- Density: 0.7
- Base Model: `meta-llama/Meta-Llama-3-8B`

In [ ]:
# Çalışma dizinini garantile
%cd {PROJECT_DIR}

# TIES merge çalıştır
!python scripts/run_merge.py --strategy ties

In [ ]:
# TIES modelini Drive'a kopyala
if MOUNT_DRIVE:
    import shutil
    from tqdm import tqdm
    import os

    src = f"{PROJECT_DIR}/merged_models/ties"
    dst = "/content/drive/MyDrive/turkish_merges/ties"

    if os.path.exists(src):
        files = [f for f in os.listdir(src) if os.path.isfile(os.path.join(src, f))]
        for f in tqdm(files, desc="TIES → Drive"):
            shutil.copy2(os.path.join(src, f), os.path.join(dst, f))
        print(f"✅ TIES modeli Drive'a kopyalandı: {dst}")
    else:
        print(f"❌ Kaynak bulunamadı: {src}")
else:
    print("ℹ️  Drive mount edilmediği için kopyalama atlandı.")

---
## [Section 4] DARE Merge
⏱ Bu section ~60 dakika sürer (T4 GPU)

DARE-TIES (Drop And REscale + TIES sign election) ile üç model birleştirilir:
- Weights: [0.4, 0.35, 0.25]
- Density: 0.5
- Base Model: `meta-llama/Meta-Llama-3-8B`

In [ ]:
# Çalışma dizinini garantile
%cd {PROJECT_DIR}

# DARE merge çalıştır
!python scripts/run_merge.py --strategy dare

In [ ]:
# DARE modelini Drive'a kopyala
if MOUNT_DRIVE:
    import shutil
    from tqdm import tqdm
    import os

    src = f"{PROJECT_DIR}/merged_models/dare"
    dst = "/content/drive/MyDrive/turkish_merges/dare"

    if os.path.exists(src):
        files = [f for f in os.listdir(src) if os.path.isfile(os.path.join(src, f))]
        for f in tqdm(files, desc="DARE → Drive"):
            shutil.copy2(os.path.join(src, f), os.path.join(dst, f))
        print(f"✅ DARE modeli Drive'a kopyalandı: {dst}")
    else:
        print(f"❌ Kaynak bulunamadı: {src}")
else:
    print("ℹ️  Drive mount edilmediği için kopyalama atlandı.")

---
## [Section 5] Benchmark
⏱ Bu section ~90 dakika sürer (T4 GPU, 4 model sırayla)

Her model için:
1. Türkçe perplexity (mc4/tr, 500 örnek)
2. 20 Türkçe soru değerlendirmesi (genel bilgi, matematik, gramer, instruction)

In [ ]:
# Çalışma dizinini garantile
%cd {PROJECT_DIR}

# Tüm modelleri benchmark et
!python scripts/benchmark.py --model all

In [ ]:
# Sonuçları güzel tablo olarak göster
import json
import os
import pandas as pd
from IPython.display import display, HTML

results_path = os.path.join(PROJECT_DIR, 'results', 'benchmark_results.json')

if not os.path.exists(results_path):
    print("⚠️  benchmark_results.json bulunamadı!")
    print("    Önce Section 5'teki benchmark hücresini çalıştırın.")
    print(f"    Aranan yol: {results_path}")
else:
    with open(results_path, 'r', encoding='utf-8') as f:
        results = json.load(f)

    # Tablo oluştur
    rows = []
    for model_key, data in results.get('models', {}).items():
        strategy_names = {
            'slerp': 'SLERP', 'ties': 'TIES',
            'dare': 'DARE', 'baseline': 'Baseline'
        }
        rows.append({
            'Model': strategy_names.get(model_key, model_key),
            'Strateji': model_key.upper(),
            'Perplexity ↓': data.get('perplexity', 'N/A'),
            'Manuel Skor /20': data.get('manual_score', 'N/A'),
        })

    df = pd.DataFrame(rows)

    # En iyi modeli bul ve vurgula
    def highlight_best(df):
        styles = pd.DataFrame('', index=df.index, columns=df.columns)
        # En düşük perplexity
        ppl_vals = pd.to_numeric(df['Perplexity ↓'], errors='coerce')
        if ppl_vals.notna().any():
            best_ppl = ppl_vals.idxmin()
            styles.loc[best_ppl, 'Perplexity ↓'] = 'background-color: #2ecc71; color: white; font-weight: bold'
        return styles

    display(df.style.apply(highlight_best, axis=None)
            .set_caption("🏆 Benchmark Sonuçları")
            .set_table_styles([{'selector': 'caption', 'props': [('font-size', '18px'), ('font-weight', 'bold')]}]))

    print(f"\n📅 Timestamp: {results.get('timestamp', 'N/A')}")

In [ ]:
# Detaylı soru-yanıt sonuçları
import json, os

results_path = os.path.join(PROJECT_DIR, 'results', 'benchmark_results.json')

if not os.path.exists(results_path):
    print("⚠️  benchmark_results.json bulunamadı. Önce benchmark'ı çalıştırın.")
else:
    with open(results_path, 'r', encoding='utf-8') as f:
        results = json.load(f)

    for model_key, data in results.get('models', {}).items():
        print(f"\n{'='*70}")
        print(f"📋 {model_key.upper()} — Detaylı Yanıtlar")
        print(f"{'='*70}")

        responses = data.get('responses', {})
        if isinstance(responses, dict) and 'error' not in responses:
            for qid, resp in responses.items():
                score_emoji = '✅' if resp.get('score', 0) == 1 else '❌'
                print(f"\n  {score_emoji} [{qid}] {resp.get('question', '?')}")
                print(f"     → {resp.get('response', 'N/A')[:200]}")
        else:
            print(f"  ⚠️  Yanıt verisi yok veya hata: {responses}")

---
## [Section 6] HuggingFace'e Push
⏱ Bu section ~30 dakika sürer (model başına ~10 dakika)

Her merge stratejisinin sonucunu HuggingFace'e yükler:
1. `Cosmobillian/TR-Llama-8B-Cosmos-Trendyol_SLERP_v1`
2. `Cosmobillian/TR-Llama-8B-3way_TIES_v1`
3. `Cosmobillian/TR-Llama-8B-3way_DARE_v1`

In [ ]:
# Çalışma dizinini garantile
%cd {PROJECT_DIR}

# Push konfigürasyonları
push_configs = [
    {
        "model_path": "./merged_models/slerp",
        "repo_id": "Cosmobillian/TR-Llama-8B-Cosmos-Trendyol_SLERP_v1",
        "strategy": "SLERP",
        "source_models": "ytu-ce-cosmos/Turkish-Llama-8b-Instruct-v0.1,Trendyol/Trendyol-LLM-8b-chat-v2.0",
    },
    {
        "model_path": "./merged_models/ties",
        "repo_id": "Cosmobillian/TR-Llama-8B-3way_TIES_v1",
        "strategy": "TIES",
        "source_models": "ytu-ce-cosmos/Turkish-Llama-8b-Instruct-v0.1,Trendyol/Trendyol-LLM-8b-chat-v2.0,malhajar/Mistral-7b-tr",
    },
    {
        "model_path": "./merged_models/dare",
        "repo_id": "Cosmobillian/TR-Llama-8B-3way_DARE_v1",
        "strategy": "DARE",
        "source_models": "ytu-ce-cosmos/Turkish-Llama-8b-Instruct-v0.1,Trendyol/Trendyol-LLM-8b-chat-v2.0,malhajar/Mistral-7b-tr",
    },
]

push_results = []

for cfg in push_configs:
    import os
    if not os.path.exists(cfg['model_path']):
        print(f"⚠️  {cfg['repo_id']}: Model bulunamadı ({cfg['model_path']}), atlanıyor.")
        push_results.append((cfg['repo_id'], '❌ Model yok'))
        continue

    print(f"\n{'='*70}")
    print(f"🚀 Push: {cfg['repo_id']}")
    print(f"{'='*70}")

    cmd = (
        f"python scripts/push_to_hub.py "
        f"--model_path {cfg['model_path']} "
        f"--repo_id {cfg['repo_id']} "
        f"--strategy {cfg['strategy']} "
        f'--source_models "{cfg["source_models"]}"'
    )

    ret = os.system(cmd)

    if ret == 0:
        url = f"https://huggingface.co/{cfg['repo_id']}"
        push_results.append((cfg['repo_id'], f'✅ {url}'))
        print(f"\n✅ Başarılı: {url}")
    else:
        push_results.append((cfg['repo_id'], '❌ Hata'))
        print(f"\n❌ Push başarısız: {cfg['repo_id']}")

# Özet
print(f"\n\n{'='*70}")
print("📋 PUSH ÖZETİ")
print(f"{'='*70}")
for repo, status in push_results:
    print(f"  {status} — {repo}")

---
## 🎉 Tamamlandı!

Tüm merge, benchmark ve push işlemleri tamamlandı.

**Sonraki adımlar:**
- HuggingFace'teki model card'ları kontrol edin
- Benchmark sonuçlarını `results/benchmark_results.json`'dan inceleyin
- Farklı parametrelerle (t, density, weights) tekrar deneyin

📂 **GitHub:** [github.com/Cosmobillian/turkish-llm-merging](https://github.com/Cosmobillian/turkish-llm-merging)